# ViGoEmotions — GRU và PhoBERT trên Colab GPU
**Runtime → Change runtime type → T4 GPU**. Upload gói `vigoemotions_colab.zip` do project tạo. Sau đó upload riêng ba CSV có quyền sử dụng. Không chia sẻ notebook có dữ liệu thô; code chỉ xuất số liệu tổng hợp và ID.

Chạy từ trên xuống. Checkpoint neural và vocab nằm trong `data/` (private). Test không dùng cho model selection. Notebook đã kiểm tra cấu trúc/cú pháp, chưa huấn luyện GPU trong phiên tạo nội dung này.

In [1]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # Chọn vigoemotions_colab.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/ViGoEmotions')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (ROOT/item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(ROOT)
os.chdir(ROOT)

ModuleNotFoundError: No module named 'google'

In [ ]:
%pip install -q -r requirements-colab.txt
!apt-get -qq update
!apt-get -qq install -y default-jre-headless fonts-dejavu-core

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi chạy tiếp.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java','-version'], check=True)

## Nạp dữ liệu được cấp phép
Chọn đủ `train.csv`, `val.csv`, `test.csv`. Dữ liệu nằm trong runtime riêng và không đưa vào gói kết quả tải xuống.

In [ ]:
uploaded = files.upload()
Path('data').mkdir(exist_ok=True)
for name in ('train.csv','val.csv','test.csv'):
    if name not in uploaded:
        raise FileNotFoundError(f'Thiếu {name}')
    Path('data',name).write_bytes(uploaded[name])
del uploaded

In [ ]:
import sys
sys.path.insert(0, str(ROOT/'scripts'))
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
subprocess.run([sys.executable,'scripts/analyze_data.py'],check=True)
subprocess.run([sys.executable,'scripts/train_tfidf.py'],check=True)

## 1. Recurrent baseline: BiGRU
Embedding 128, hidden 128 mỗi hướng, dropout 0.3, 10 epoch, batch 64. Chọn epoch theo validation Macro-F1 tại ngưỡng 0.5; không early-stop bằng test.

In [ ]:
subprocess.run([sys.executable,'scripts/train_neural.py','--model','gru'],check=True)

## 2. PhoBERT
Checkpoint/revision được khóa trong configs/midterm.json. Dùng VnCoreNLP để word segmentation trước BPE theo model card. Classification head 27 logits, BCEWithLogitsLoss, sigmoid khi suy luận. Lần chạy đầu tải checkpoint và tài nguyên segmenter từ mạng.

In [ ]:
import py_vncorenlp
segmenter_dir = (ROOT/'data/vncorenlp').resolve()
segmenter_dir.mkdir(parents=True,exist_ok=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
subprocess.run([sys.executable,'scripts/train_neural.py','--model','phobert','--vncorenlp-dir',str(segmenter_dir)],check=True)

## 3. So sánh và xuất báo cáo
Báo cáo lấy số liệu từ các file kết quả hiện có. Câu hỏi PhoBERT vs GRU được tính trên cùng nhóm validation; nhớ đọc giới hạn heuristic và cỡ mẫu.

In [ ]:
subprocess.run([sys.executable,'scripts/build_report.py'],check=True)
import pandas as pd
from IPython.display import display
display(pd.read_csv('reports/tables/model_comparison.csv'))
display(pd.read_csv('reports/tables/neural_style_comparison.csv'))

In [ ]:
import zipfile
with zipfile.ZipFile('/content/vigoemotions_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('reports','splits'):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p,str(p))
files.download('/content/vigoemotions_results.zip')

## Bảo toàn checkpoint nếu cần tiếp tục Final
Các checkpoint nằm ở `data/checkpoints/`. Lưu vào Drive riêng trước khi runtime hết hạn. Gói kết quả chỉ chứa reports/splits; **không chứa trọng số, vocab hoặc comment thô**. Mỗi run JSON ghi config, package versions, GPU, thời gian, seed và epoch tốt nhất.

Tài liệu: https://huggingface.co/vinai/phobert-base-v2 ; https://github.com/vncorenlp/VnCoreNLP ; https://aclanthology.org/2026.eacl-long.129/